In [104]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
import warnings

warnings.filterwarnings("ignore")


In [105]:
x="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataRestaurantCPI/UEMOA"

In [106]:
# 1. Charger le fichier CSV

df = pd.read_excel(x+".xlsx") 

# Vérifier les valeurs manquantes
print("Valeurs manquantes par colonne :")
print(df.isna().sum())
print("\nAperçu des données :")
print(df.head())

Valeurs manquantes par colonne :
Months                                      0
Restauration CPI                            0
Natural gas, US ($/mmbtu) NGAS_US           0
Natural gas, Europe ($/mmbtu) NGAS_EUR      0
Cocoa ($/kg) COCOA                          0
Coffee, Arabica ($/kg) COFFEE_ARABIC        0
Coffee, Robusta ($/kg) COFFEE_ROBUS         0
Tea, avg 3 auctions ($/kg) TEA_AVG          0
Tea, Colombo ($/kg) TEA_COLOMBO             0
Tea, Kolkata ($/kg) TEA_KOLKATA             0
Tea, Mombasa ($/kg) TEA_MOMBASA             0
Coconut oil ($/mt) COCONUT_OIL              0
Groundnuts ($/mt) GRNUT                     0
Fish meal ($/mt) FISH_MEAL                  0
Groundnut oil **($/mt) GRNUT_OIL            0
Palm oil ($/mt) PALM_OIL                    0
Palm kernel oil ($/mt) PLMKRNL_OIL          0
Soybeans ($/mt) SOYBEANS                    0
Soybean oil ($/mt) SOYBEAN_OIL              0
Soybean meal ($/mt) SOYBEAN_MEAL            0
Rapeseed oil ($/mt) RAPESEED_OIL            0
S

In [107]:
# 2. Fonction pour choisir le meilleur nombre de lags

def meilleur_lag(serie, max_lags=10):
    serie = serie.dropna()
    if len(serie) < 5:  # Trop peu de points pour un modèle AR
        return 1
    aic_values = {}
    for lag in range(1, min(max_lags, len(serie)//2)):
        try:
            model = AutoReg(serie, lags=lag, old_names=False)
            result = model.fit()
            aic_values[lag] = result.aic
        except:
            continue
    return min(aic_values, key=aic_values.get) if aic_values else 1


# 3. Fonction pour remplir les NaN avec AR

def Interpol_AR_backcaster(serie):
    if serie.isna().sum() == len(serie):
        return serie
    
    # Nombre de valeurs manquantes au début
    backcast_steps = 0
    for val in serie:
        if pd.isna(val):
            backcast_steps += 1
        else:
            break
    
    # Interpolation initiale
    serie_init = serie.interpolate(method="linear", limit_direction="both")
    
    # Déterminer le meilleur lag
    lags = meilleur_lag(serie_init)
    
    # Entraîner AR
    model = AutoReg(serie_init, lags=lags, old_names=False)
    model_fit = model.fit()
    
    # Prédire toute la série existante
    prediction = model_fit.predict(start=0, end=len(serie_init)-1)
    
    # Remplacer les NaN internes
    serie_finale = serie.copy()
    serie_finale[serie_finale.isna()] = prediction[serie_finale.isna()]
    
    # 🔄 Backcasting (uniquement si on a des NaN au début)
    backcast_vals = []
    history = serie_finale.dropna().tolist()
    
    for i in range(backcast_steps):
        coeffs = model_fit.params
        lags_values = history[:lags]  # premiers points connus
        back_val = np.mean(lags_values) if lags_values else history[0]
        backcast_vals.append(back_val)
    
    # Ajouter les valeurs estimées au début
    backcast_vals = backcast_vals[::-1]  # ordre chronologique
    serie_complete = pd.Series(backcast_vals + serie_finale.tolist())
    
    return serie_complete


In [108]:
# 4. Appliquer la fonction à toutes les colonnes numériques

df_complet = df.copy()

for col in df.columns:
    if df[col].dtype in [np.float64, np.int64]:  
        print(f"🔄 Traitement de la colonne : {col}")
        
        # Tant qu'il reste des valeurs manquantes, on continue
        iteration = 0
        while df_complet[col].isna().sum() > 0:
            iteration += 1
            print(f"   ➝ Itération {iteration} : {df_complet[col].isna().sum()} valeurs manquantes")
            
            df_complet[col] = Interpol_AR_backcaster(df_complet[col])
            
            # Sécurité : éviter boucle infinie si jamais ça bloque
            if iteration > 10:
                print(f"⚠️  Trop d'itérations pour la colonne {col}, arrêt forcé.")
                break
            


🔄 Traitement de la colonne : Restauration CPI
🔄 Traitement de la colonne : Natural gas, US ($/mmbtu) NGAS_US
🔄 Traitement de la colonne : Natural gas, Europe ($/mmbtu) NGAS_EUR
🔄 Traitement de la colonne : Cocoa ($/kg) COCOA
🔄 Traitement de la colonne : Coffee, Arabica ($/kg) COFFEE_ARABIC
🔄 Traitement de la colonne : Coffee, Robusta ($/kg) COFFEE_ROBUS
🔄 Traitement de la colonne : Tea, avg 3 auctions ($/kg) TEA_AVG
🔄 Traitement de la colonne : Tea, Colombo ($/kg) TEA_COLOMBO
🔄 Traitement de la colonne : Tea, Kolkata ($/kg) TEA_KOLKATA
🔄 Traitement de la colonne : Tea, Mombasa ($/kg) TEA_MOMBASA
🔄 Traitement de la colonne : Coconut oil ($/mt) COCONUT_OIL
🔄 Traitement de la colonne : Groundnuts ($/mt) GRNUT
🔄 Traitement de la colonne : Fish meal ($/mt) FISH_MEAL
🔄 Traitement de la colonne : Groundnut oil **($/mt) GRNUT_OIL
🔄 Traitement de la colonne : Palm oil ($/mt) PALM_OIL
🔄 Traitement de la colonne : Palm kernel oil ($/mt) PLMKRNL_OIL
🔄 Traitement de la colonne : Soybeans ($/mt) SOY

🔄 Traitement de la colonne : Wheat, US SRW ($/mt) WHEAT_US_SRW
🔄 Traitement de la colonne : Wheat, US HRW ($/mt) WHEAT_US_HRW
   ➝ Itération 1 : 58 valeurs manquantes
   ➝ Itération 2 : 9 valeurs manquantes
🔄 Traitement de la colonne : Banana, Europe ($/kg) BANANA_EU
   ➝ Itération 1 : 7 valeurs manquantes
🔄 Traitement de la colonne : Banana, US ($/kg) BANANA_US
🔄 Traitement de la colonne : Orange ($/kg) ORANGE
🔄 Traitement de la colonne : Beef **($/kg) BEEF
🔄 Traitement de la colonne : Chicken **($/kg) CHICKEN
🔄 Traitement de la colonne : Lamb **($/kg) LAMB
🔄 Traitement de la colonne : Shrimps, Mexican ($/kg) SHRIMP_MEX
🔄 Traitement de la colonne : Sugar, EU ($/kg) SUGAR_EU
🔄 Traitement de la colonne : Sugar, US ($/kg) SUGAR_US
   ➝ Itération 1 : 20 valeurs manquantes
🔄 Traitement de la colonne : Sugar, world ($/kg) SUGAR_WLD
🔄 Traitement de la colonne : Unnamed: 41
🔄 Traitement de la colonne : Unnamed: 42


In [109]:

# Vérifier si toutes les NaN ont disparu
print("\nValeurs manquantes après traitement :")
print(df_complet.isna().sum())

# 5. Sauvegarder le fichier complété
df_complet.to_excel(x+"_completes.xlsx", index=False)
print("\n✅ Fichier Excel complété enregistré :", x+"_completes.xlsx")



Valeurs manquantes après traitement :
Months                                     0
Restauration CPI                           0
Natural gas, US ($/mmbtu) NGAS_US          0
Natural gas, Europe ($/mmbtu) NGAS_EUR     0
Cocoa ($/kg) COCOA                         0
Coffee, Arabica ($/kg) COFFEE_ARABIC       0
Coffee, Robusta ($/kg) COFFEE_ROBUS        0
Tea, avg 3 auctions ($/kg) TEA_AVG         0
Tea, Colombo ($/kg) TEA_COLOMBO            0
Tea, Kolkata ($/kg) TEA_KOLKATA            0
Tea, Mombasa ($/kg) TEA_MOMBASA            0
Coconut oil ($/mt) COCONUT_OIL             0
Groundnuts ($/mt) GRNUT                    0
Fish meal ($/mt) FISH_MEAL                 0
Groundnut oil **($/mt) GRNUT_OIL           0
Palm oil ($/mt) PALM_OIL                   0
Palm kernel oil ($/mt) PLMKRNL_OIL         0
Soybeans ($/mt) SOYBEANS                   0
Soybean oil ($/mt) SOYBEAN_OIL             0
Soybean meal ($/mt) SOYBEAN_MEAL           0
Rapeseed oil ($/mt) RAPESEED_OIL           0
Sunflower oil ($